# HDBSCAN Clustering Pipeline v2

**Author:** Nadia ELORGA-CASTAGNET  
**CHU de Bordeaux - Master 2 PHDS 2025/2026**

## Pipeline overview

```
CHUNK 1 - Configuration, imports, column definitions
CHUNK 2 - All functions
CHUNK 3 - STEP 1: Distance matrix + 2D grid sweep (mcs x min_samples)
CHUNK 4 - STEP 2: Final run with full visualizations
```

## Key changes from v1
- **2D grid sweep**: sweeps mcs AND min_samples simultaneously, all values as % of N
- min_samples always << mcs -> fewer outliers
- 4 heatmaps per run: outlier rate, silhouette, combined score, n_clusters
- cluster_selection_method = 'leaf' available as alternative to 'eom'
- All hyperparameter values derived from N -> portable to N=120000 without code changes


---
## CHUNK 1 - Configuration, Imports & Column Definitions


In [37]:
# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================

CSV_PATH   = "df_final_binaire_imputed.csv"
OUTPUT_DIR = "Results/Regular_clustering/Full_dataset/With_counts"

GPU_DEVICE_ID = 1

SCALERS = ["minmax", "standard"]

# Hyperparameters for UMAP (reduction dimension)
UMAP_N_NEIGHBORS  = 30
UMAP_MIN_DIST     = 0.0
UMAP_N_COMPONENTS = 10
UMAP_RANDOM_STATE = 42

# Hyperparameters for UMAP & TSNE (visualization)
VIZ_N_NEIGHBORS = 30
VIZ_MIN_DIST    = 0.1
VIZ_TSNE_PERP   = 30
VIZ_TSNE_ITER   = 1500

# 'eom'  = Excess of Mass (fewer clusters, more outliers)
# 'leaf' = leaf clusters (more clusters, fewer outliers) <- try this to reduce outliers
HDBSCAN_CLUSTER_METHOD = "eom"

# 2D Grid sweep - parametrized as % of N
# mcs: 0.5% to 5% of N (minimum cluster size)
# ms : 1% to 20% of mcs (local density - always much smaller than mcs -> fewer outliers)
MCS_PCT_VALUES    = [0.02, 0.03, 0.04, 0.05, 0.06]
MS_PCT_MCS_VALUES = [0.01, 0.05, 0.10, 0.20]
MS_FIXED_VALUES   = [1, 5, 10, 15]





# ------------------------------------------------------------------------------------------

# Combined score weight


N_MIN_CLUSTERS = 5
N_MAX_CLUSTERS = 15

W_SILHOUETTE = 0.35
W_STABILITY  = 0.25
W_OUTLIER    = 0.25
W_NCLUSTERS  = 0.15


# ------------------------------------------------------------------------------------------

#WEIGHT_HOSP_S2          = 3.0
#WEIGHT_BIO_VEINOUS      = 1 / 27
#WEIGHT_IMAGING_DETAILED = 1 / 10
#WEIGHT_HOSP_S3          = 5.0

# Normalization weighting each bloc = to 1
WEIGHT_BIO_VEINOUS      = 1 / 27   # bloc bio veineux (27 vars) → total = 1
WEIGHT_IMAGING_DETAILED = 1 / 10   # bloc imagerie détaillée (10 vars) → total = 1
WEIGHT_IMAGING_BOOL     = 1 / 4    # bloc imagerie bool (4 vars) → total = 1
WEIGHT_BIO_EXAMS        = 1 / 4    # bloc labo broad (4 vars) → total = 1
WEIGHT_DISPOSITION      = 1 / 2    # bloc disposition (2 vars) → total = 1
WEIGHT_PROCEDURES = WEIGHT_BIO_EXAMS   # = 1/4 = 0.25



BIO_BINS    = [-1, 0, 1, 2, 10]
BIO_LABELS  = ["0", "1", "2", "3+"]
IMAG_BINS   = [-1, 0, 1, 2, 10]
IMAG_LABELS = ["0", "1", "2", "3+"]

# ==============================================================================
# 1. IMPORTS
# ==============================================================================

import os, logging, time, pickle
import gower, hdbscan
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import torch
import umap
import umap.umap_ as umap_reduce
from datetime import datetime
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

torch.cuda.set_device(GPU_DEVICE_ID)
torch.cuda.set_per_process_memory_fraction(0.5, device=GPU_DEVICE_ID)
print(f"GPU : {torch.cuda.get_device_name(GPU_DEVICE_ID)}")
print(f"Available memory : {torch.cuda.get_device_properties(GPU_DEVICE_ID).total_memory / 1e9:.0f} GB")

# ==============================================================================
# 2. COLUMN DEFINITIONS
# ==============================================================================

IMAGING_COLS_BOOL = {"has_ultrasound", "has_ct_scan", "has_xray", "has_mri"}

IMAGING_COLS_DETAILED = {
    "ultrasound_1", "ultrasound_2",
    "ct_scan_1", "ct_scan_2", "ct_scan_3",
    "xray_1", "xray_2", "xray_3",
    "mri_1", "mri_2",
}

BIO_VEINOUS = {
    "is_hemoglobine", "is_leucocytes", "is_formule_leuco",
    "is_urea", "is_creatinine", "is_sodium", "is_potassium",
    "is_platelets", "is_pt", "is_aptt", "is_calcium", "is_ck",
    "is_lactates", "is_troponine", "is_bnp", "is_ckmb", "is_ddimer",
    "is_crp", "is_pct", "is_alat", "is_asat", "is_bili_total",
    "is_lipase", "is_alp",
    "is_calcium_ionized", "is_aXa_aIIa", "is_fibrinogen",
}

BIO_EXAMS        = {"has_blood_test", "has_culture", "has_lumbar_puncture", "has_blood_gas"}
PROCEDURE_COLS   = {"had_ekg"}
DISPOSITION_COLS = {"hospitalization", "observation_unit",
                    #"inter_facility_transfer"
                    }
COLS_QUANTI      = ["imaging_exam_count", "bio_exam_count"]

SCENARIOS = {
    "scenario_2": list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS) + COLS_QUANTI,
    "scenario_3": list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS
                       | IMAGING_COLS_DETAILED | BIO_VEINOUS) + COLS_QUANTI,
}

print(f"Scenario 2: {len(SCENARIOS['scenario_2'])} variables")
print(f"Scenario 3: {len(SCENARIOS['scenario_3'])} variables")


GPU : NVIDIA A100-SXM4-80GB
Available memory : 85 GB
Scenario 2: 13 variables
Scenario 3: 50 variables


eom — Excess of Mass (valeur par défaut)
Il sélectionne les clusters en maximisant la stabilité de chaque nœud de l'arbre. Un cluster "vit" aussi longtemps que possible avant de se fragmenter.

→ Clusters de tailles variées, potentiellement très inégaux
→ Peut donner quelques gros clusters + des petits clusters nichés à l'intérieur
→ Plus fidèle à la structure hiérarchique réelle des données
→ Favorise des clusters compacts et denses


leaf
Il sélectionne uniquement les feuilles de la condensed tree, donc les clusters les plus petits et les plus "fins" possibles.

→ Clusters plus nombreux et plus petits
→ Tailles plus homogènes
→ Ignore la hiérarchie → découpe toujours au niveau le plus fin
→ Peut fragmenter des groupes qui auraient naturellement dû rester ensemble

---
## CHUNK 2 - All Functions


In [38]:
# ==============================================================================
# FUNCTION 1: Data loading
# ==============================================================================

def load_and_preprocess(csv_path=CSV_PATH):
    df = pd.read_csv(csv_path, low_memory=False)
    df["bio_exam_cat"]     = pd.cut(df["bio_exam_count"],     bins=BIO_BINS,  labels=BIO_LABELS)
    df["imaging_exam_cat"] = pd.cut(df["imaging_exam_count"], bins=IMAG_BINS, labels=IMAG_LABELS)
    for col in ("bio_exam_count", "imaging_exam_count"):
        dist = (df[col].value_counts(dropna=False).sort_index()
                .rename_axis(col).reset_index(name="n"))
        dist["pct"] = (dist["n"] / dist["n"].sum() * 100).round(1)
        log.info(f"\n{col} distribution:\n{dist.to_string(index=False)}")
    return df


# ==============================================================================
# FUNCTION 2: GPU distance matrix (Scenario 2)
# Hamming for binary (double-zeros count as similarity).
# Manhattan for quantitative after scaling.
# ==============================================================================

def save_matrix(arr: np.ndarray, sc_dir: str, name: str, key: str, run_label: str = "") -> str:
    timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
    label_part  = f"_{run_label}" if run_label else ""
    latest_path = os.path.join(sc_dir, f"{name}{label_part}_latest.npz")
    np.savez_compressed(latest_path, **{key: arr})
    size_mb = os.path.getsize(latest_path) / 1e6
    log.info(f"Saved {name}{label_part} [{timestamp}]: {latest_path}  ({size_mb:.1f} MB)")
    return latest_path


def compute_distance_matrix_cpu(df_sub, binary_cols, cat_cols, quanti_cols,
                                  col_weights=None,
                                  weight_quanti=1.0,
                                  scaler="minmax"):
    from sklearn.preprocessing import MinMaxScaler, StandardScaler
    from sklearn.metrics import pairwise_distances

    n = len(df_sub)
    D = np.zeros((n, n), dtype=np.float32)

    # ── Binary (Hamming) ─────────────────────────────────────────────────────
    for col in binary_cols:
        vals = df_sub[col].values.reshape(-1, 1).astype("float32")
        d    = pairwise_distances(vals, metric="manhattan")
        w    = col_weights.get(col, 1.0) if col_weights else 1.0
        D   += d * w

    # ── Categorical (Hamming) ─────────────────────────────────────────────────
    for col in cat_cols:
        codes = df_sub[col].cat.codes.values.reshape(-1, 1).astype("float32")
        w     = col_weights.get(col, 1.0) if col_weights else 1.0
        D    += pairwise_distances(codes, metric="hamming") * len(df_sub) * w

    # ── Quantitative (Manhattan) ──────────────────────────────────────────────
    if quanti_cols:
        q        = df_sub[quanti_cols].values.astype("float32")
        q_scaled = (MinMaxScaler().fit_transform(q) if scaler == "minmax"
                    else StandardScaler().fit_transform(q)).astype("float32")
        D       += pairwise_distances(q_scaled, metric="manhattan") * weight_quanti

    np.fill_diagonal(D, 0)
    return D.astype(np.float64)

# def compute_distance_matrix_gpu(df_sub, binary_cols, cat_cols, quanti_cols,
#                                   weight_hosp=1.0, weight_quanti=1.0,
#                                   scaler="minmax", device_id=GPU_DEVICE_ID):   # ← retiré sc_dir et save
#     device = torch.device(f"cuda:{device_id}")
#     n      = len(df_sub)
#     D      = torch.zeros((n, n), device=device, dtype=torch.float32)
#
#     for col in binary_cols:
#         vals = torch.tensor(df_sub[col].values, dtype=torch.float32, device=device).unsqueeze(1)
#         d    = torch.cdist(vals, vals, p=1)
#         w    = weight_hosp if col == "hospitalization" else 1.0
#         D   += d * w
#
#     for col in cat_cols:
#         codes = torch.tensor(df_sub[col].cat.codes.values, dtype=torch.float32, device=device).unsqueeze(1)
#         D    += (codes != codes.T).float()
#
#     if quanti_cols:
#         q        = df_sub[quanti_cols].values.astype("float32")
#         q_scaled = (MinMaxScaler().fit_transform(q) if scaler == "minmax"
#                     else StandardScaler().fit_transform(q)).astype("float32")
#         q_tensor = torch.tensor(q_scaled, dtype=torch.float32, device=device)
#         D       += torch.cdist(q_tensor, q_tensor, p=1) * weight_quanti
#
#     D.fill_diagonal_(0)
#     return D.cpu().numpy().astype(np.float64)

# ==============================================================================
# FUNCTION 3: Main HDBSCAN run
# Returns: (df_sub, D, fit_input, labels, clusterer)
# ==============================================================================

def run_hdbscan(df, scenario_name, run_label, distance_metric,
                scaler="minmax", col_weights=None, weight_quanti=1.0,
                gower_weights=None, min_cluster_size=500, min_samples=10):

    sc_dir = os.path.join(OUTPUT_DIR,
                          scaler if distance_metric == "precomputed" else "gower",
                          run_label)
    os.makedirs(sc_dir, exist_ok=True)

    cols   = [c for c in SCENARIOS[scenario_name] if c in df.columns]
    df_sub = df[cols].copy()

    if scenario_name == "scenario_3":
        essential = [c for c in cols if c not in IMAGING_COLS_DETAILED and c not in BIO_VEINOUS]
        df_sub    = df_sub.dropna(subset=essential)
    else:
        df_sub = df_sub.dropna()

    idx         = df_sub.index
    quanti_cols = [c for c in COLS_QUANTI if c in df_sub.columns]
    binary_cols = [c for c in df_sub.columns
                   if c not in quanti_cols and set(df_sub[c].dropna().unique()) <= {0, 1}]
    cat_cols    = [c for c in df_sub.columns if c not in binary_cols and c not in quanti_cols]
    for col in cat_cols:
        df_sub[col] = df_sub[col].astype("category")

    log.info(f"[{run_label}] n={len(df_sub)} | binary={len(binary_cols)} | cat={len(cat_cols)} | quanti={len(quanti_cols)}")

    t0       = time.time()
    col_list = list(df_sub.columns)
    n_vars   = len(col_list)

    if distance_metric == "gower":
        df_gower = df_sub.copy()
        for col in df_gower.select_dtypes(include="integer").columns:
            df_gower[col] = df_gower[col].astype("float64")
        for col in df_gower.select_dtypes(include="category").columns:
            df_gower[col] = df_gower[col].astype("object")
        weights = np.ones(n_vars)
        if gower_weights:
            for col, w in gower_weights.items():
                if col in col_list:
                    weights[col_list.index(col)] = w
        weights        = weights / weights.sum() * n_vars
        D              = gower.gower_matrix(df_gower, weight=weights).astype(np.float64)
        np.fill_diagonal(D, 0)
        fit_input      = D
        hdbscan_metric = "precomputed"
        if scenario_name != "scenario_3":
            save_matrix(D, sc_dir, name="distance_matrix", key="D",
                        run_label=run_label)

    else:
        D = compute_distance_matrix_cpu(
            df_sub, binary_cols=binary_cols, cat_cols=cat_cols,
            quanti_cols=quanti_cols, col_weights=col_weights,
            weight_quanti=weight_quanti, scaler=scaler,
        )
        fit_input      = D
        hdbscan_metric = "precomputed"
        if scenario_name != "scenario_3":
            save_matrix(D, sc_dir, name="distance_matrix", key="D",
                        run_label=run_label)

    log.info(f"[{run_label}] Distance matrix: {time.time()-t0:.1f}s")

    if scenario_name == "scenario_3":
        t0      = time.time()
        reducer = umap_reduce.UMAP(
            n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
            n_components=UMAP_N_COMPONENTS, metric="precomputed",
            random_state=UMAP_RANDOM_STATE,
        )
        emb = reducer.fit_transform(D)
        save_matrix(emb, sc_dir, name="umap_embedding", key="embedding",
                    run_label=run_label)
        log.info(f"[{run_label}] UMAP shape={emb.shape} - {time.time()-t0:.1f}s")
        fit_input      = emb
        hdbscan_metric = "euclidean"

    t0        = time.time()

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size, min_samples=min_samples,
        metric= hdbscan_metric, cluster_selection_method=HDBSCAN_CLUSTER_METHOD,
        gen_min_span_tree=True,
    ).fit(fit_input)
    log.info(f"[{run_label}] HDBSCAN: {time.time()-t0:.1f}s")

    labels     = clusterer.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    log.info(f"[{run_label}] clusters={n_clusters} | noise={n_noise} ({100*n_noise/len(labels):.1f}%)")

    df_out            = df.loc[idx].copy()
    df_out["cluster"] = labels
    df_out.to_csv(os.path.join(sc_dir, f"clustering_mcs{min_cluster_size}_ms{min_samples}.csv"), index=False)

    return df_sub, D, fit_input, labels, clusterer


# ==============================================================================
# FUNCTION 4: 2D Grid sweep - mcs x min_samples as % of N
#
# Key principle: min_samples << mcs
#   Large min_samples -> stricter density -> MORE outliers
#   Small min_samples -> looser density  -> FEWER outliers
#
# All values derived from N -> same code works for N=30000 and N=120000
# ==============================================================================

def make_mcs_ms_grid(N):
    grid = []
    for mcs_pct in MCS_PCT_VALUES:
        mcs = max(10, int(N * mcs_pct))
        ms_values = sorted(set(MS_FIXED_VALUES + [max(1, int(mcs * p)) for p in MS_PCT_MCS_VALUES]))
        for ms in ms_values:
            grid.append({
                "mcs":        mcs,
                "ms":         ms,
                "mcs_pct_N":  round(mcs / N * 100, 2),
                "ms_pct_mcs": round(ms / mcs * 100, 2),
            })
    return grid



def cluster_score(n, n_min=N_MIN_CLUSTERS, n_max=N_MAX_CLUSTERS):
    if n_min <= n <= n_max:
        return 1.0
    elif n < n_min:
        return max(0.0, (n - 2) / (n_min - 2))
    else:
        return max(0.0, (50 - n) / (50 - n_max))



def run_grid_sweep(fit_input, metric, run_label, out_dir, N):
    """
    2D grid sweep over (mcs, min_samples).
    Outputs: CSV + 4 heatmaps + top-10 summary tables.
    """
    if metric == "precomputed":
        np.fill_diagonal(fit_input, 0)

    grid = make_mcs_ms_grid(N)
    print(f"\n Grid: {len(grid)} combinations for N={N:,}")
    print(f"   mcs range: {int(N*MCS_PCT_VALUES[0]):,} - {int(N*MCS_PCT_VALUES[-1]):,}")
    print(f"   ms  range: fixed {MS_FIXED_VALUES} + {[str(round(p*100))+'%mcs' for p in MS_PCT_MCS_VALUES]}")

    results = []
    for params in grid:
        mcs = params["mcs"]
        ms  = params["ms"]

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=mcs, min_samples=ms,
            metric=metric, cluster_selection_method=HDBSCAN_CLUSTER_METHOD,
            gen_min_span_tree=False,
        ).fit(fit_input)

        labels       = clusterer.labels_
        n_clusters   = len(np.unique(labels[labels >= 0]))
        outlier_rate = float(np.mean(labels == -1))
        sil = (silhouette_score(fit_input, labels, metric=metric)
               if n_clusters >= 2 else np.nan)
        stability = (float(np.mean(clusterer.cluster_persistence_))
                     if len(clusterer.cluster_persistence_) > 0 else np.nan)

        results.append({
            "mcs":          mcs,
            "mcs_pct_N":    params["mcs_pct_N"],
            "ms":           ms,
            "ms_pct_mcs":   params["ms_pct_mcs"],
            "n_clusters":   n_clusters,
            "outlier_rate": outlier_rate,
            "outlier_pct":  round(outlier_rate * 100, 1),
            "silhouette":   round(sil, 4) if not np.isnan(sil) else np.nan,
            "stability":    round(stability, 4) if not np.isnan(stability) else np.nan,
        })

        sil_str = f"{sil:.3f}" if not np.isnan(sil) else "nan"
        log.info(f"[{run_label}] mcs={mcs:5d} ({params['mcs_pct_N']:.2f}%N) | "
                 f"ms={ms:4d} ({params['ms_pct_mcs']:.1f}%mcs) | "
                 f"clusters={n_clusters} | outliers={outlier_rate*100:.1f}% | sil={sil_str}")

    df_res = pd.DataFrame(results)

    # ── Combined score ──────────────────────────────────────────────
    def _minmax(s):
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn) if mx != mn else s * 0

    df_res["combined_score"] = (
        W_SILHOUETTE * _minmax(df_res["silhouette"].fillna(0))
        + W_STABILITY  * _minmax(df_res["stability"].fillna(0))
        - W_OUTLIER    * df_res["outlier_rate"]
        + W_NCLUSTERS  * df_res["n_clusters"].apply(cluster_score)
    ).round(4)

    # ── Save CSV ────────────────────────────────────────────────────
    os.makedirs(out_dir, exist_ok=True)
    df_res.to_csv(os.path.join(out_dir, f"grid_sweep_{run_label}.csv"), index=False)


    # ── Line plots (one per mcs value, 5 curves per plot) ───────────
    def _minmax_plot(s):
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn) if mx != mn else pd.Series([0.5] * len(s), index=s.index)

    curve_colors = {
        "silhouette"  : "#2196F3",  # blue
        "stability"   : "#4CAF50",  # green
        "outlier"     : "#F44336",  # red
        "n_clusters"  : "#FF9800",  # orange
        "combined"    : "#9C27B0",  # purple
    }

    for mcs_val in sorted(df_res["mcs"].unique()):
        df_mcs   = df_res[df_res["mcs"] == mcs_val].copy().sort_values("ms")
        mcs_pct  = df_mcs["mcs_pct_N"].iloc[0]

        sil_norm = _minmax_plot(df_mcs["silhouette"].fillna(0))
        stab_norm= _minmax_plot(df_mcs["stability"].fillna(0))
        out_inv  = 1 - df_mcs["outlier_rate"]
        nc_score = df_mcs["n_clusters"].apply(cluster_score)
        comb     = _minmax_plot(df_mcs["combined_score"])

        fig, ax = plt.subplots(figsize=(10, 5), facecolor="white")
        ax.set_facecolor("white")
        ax.plot(df_mcs["ms"], sil_norm,  color=curve_colors["silhouette"],
                marker="o", linewidth=2, label="Silhouette (norm)")
        ax.plot(df_mcs["ms"], stab_norm, color=curve_colors["stability"],
                marker="s", linewidth=2, label="Stability (norm)")
        ax.plot(df_mcs["ms"], out_inv,   color=curve_colors["outlier"],
                marker="^", linewidth=2, label="1 - Outlier rate")
        ax.plot(df_mcs["ms"], nc_score,  color=curve_colors["n_clusters"],
                marker="D", linewidth=2, label="N clusters score")
        ax.plot(df_mcs["ms"], comb,      color=curve_colors["combined"],
                marker="*", linewidth=2.5, linestyle="--",
                markersize=10, label="Combined score")

        ax.set_xlabel("min_samples (ms)", fontsize=11)
        ax.set_ylabel("Score (normalized to [0, 1])", fontsize=11)
        ax.set_title(
            f"Clustering scores vs. min_samples — {run_label}\n"
            f"mcs = {mcs_val} ({mcs_pct:.2f}% of N)",
            fontsize=12, fontweight="bold"
        )
        ax.legend(loc="upper right", fontsize=9)
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, alpha=0.3)
        sns.despine(ax=ax)
        plt.tight_layout()
        plt.savefig(
            os.path.join(out_dir, f"grid_{run_label}_curves_mcs{mcs_val}.png"),
            dpi=150,
            facecolor="white"
        )
        plt.show()
        plt.close()

    return df_res


# ==============================================================================
# FUNCTION 5: HDBSCAN native tree visualizations
# Condensed tree, single linkage tree, MST (S3 only), cluster persistence
# ==============================================================================

def plot_hdbscan_trees(clusterer, run_label, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    plt.style.use("default")

    try:
        fig, ax = plt.subplots(figsize=(14, 7), facecolor="white")
        ax.set_facecolor("white")
        clusterer.condensed_tree_.plot(select_clusters=True, axis=ax, colorbar=True)
        ax.set_title(f"Condensed Tree - {run_label}")
        ax.set_xlabel("Points / clusters")
        ax.set_ylabel("lambda (stability)")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "condensed_tree.png"), dpi=150, facecolor="white")
        plt.close()
    except Exception as e:
        log.warning(f"[{run_label}] Condensed tree: {e}")

    try:
        fig, ax = plt.subplots(figsize=(14, 7), facecolor="white")
        ax.set_facecolor("white")
        clusterer.single_linkage_tree_.plot(axis=ax)
        ax.set_title(f"Single Linkage Tree - {run_label}")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "single_linkage_tree.png"), dpi=150, facecolor="white")
        plt.close()
    except Exception as e:
        log.warning(f"[{run_label}] Single linkage tree: {e}")

    try:
        mst = clusterer.minimum_spanning_tree_
        if mst is not None:
            fig, ax = plt.subplots(figsize=(14, 7), facecolor="white")
            ax.set_facecolor("white")
            mst.plot(edge_cmap="viridis", edge_alpha=0.6, node_size=10, axis=ax)
            ax.set_title(f"Minimum Spanning Tree - {run_label}")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "minimum_spanning_tree.png"), dpi=150, facecolor="white")
            plt.close()
    except Exception as e:
        log.warning(f"[{run_label}] MST: {e}")

    try:
        persistence = clusterer.cluster_persistence_
        if len(persistence) > 0:
            fig, ax = plt.subplots(figsize=(max(6, len(persistence)), 4), facecolor="white")
            ax.set_facecolor("white")
            colors = ["#e05c2e" if p == max(persistence) else "#aec6cf" for p in persistence]
            ax.bar(range(len(persistence)), persistence, color=colors)
            ax.set_xlabel("Cluster ID")
            ax.set_ylabel("Persistence (stability)")
            ax.set_title(f"Cluster Persistence - {run_label}")
            ax.set_xticks(range(len(persistence)))
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "cluster_persistence.png"), dpi=150, facecolor="white")
            plt.close()
    except Exception as e:
        log.warning(f"[{run_label}] Cluster persistence: {e}")


# ==============================================================================
# FUNCTION 6: Cluster profile heatmap + Ward dendrogram
# Heatmap: mean variable values per cluster (outliers excluded)
# Dendrogram: annotates each merge with the most discriminating variable
# ==============================================================================

def plot_cluster_heatmap(df_sub, labels, title, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    df_prof            = df_sub.copy()
    df_prof["cluster"] = labels
    df_prof            = df_prof[df_prof["cluster"] != -1]
    if df_prof.empty:
        return pd.DataFrame()
    numeric_cols = [c for c in df_prof.select_dtypes(include=["number"]).columns if c != "cluster"]
    profile      = df_prof[numeric_cols + ["cluster"]].groupby("cluster").mean().round(3)
    if profile.empty:
        return profile
    n_cols = profile.shape[1]
    n_rows = profile.shape[0]
    fig, ax = plt.subplots(figsize=(max(8, n_cols * 0.8), max(4, n_rows * 0.6)), facecolor="white")
    ax.set_facecolor("white")
    sns.heatmap(profile, annot=True, cmap="YlOrRd", fmt=".2f",
                annot_kws={"size": max(6, min(10, 80 // n_cols))}, ax=ax)
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    log.info(f"Saved: {path}")
    return profile

def plot_cluster_dendrogram(profile, title, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)

    # Normalize profiles so all variables are comparable
    profile_scaled = pd.DataFrame(
        StandardScaler().fit_transform(profile),
        index   = profile.index,
        columns = profile.columns,
    )

    Z         = linkage(profile_scaled.values, method="ward")
    variables = profile_scaled.columns.tolist()

    # Pre-compute most discriminating variable at each merge
    cluster_vectors = {i: profile_scaled.iloc[i].values for i in range(len(profile_scaled))}
    top_vars        = []
    for i, (c1, c2, dist, _) in enumerate(Z):
        c1, c2   = int(c1), int(c2)
        v1, v2   = cluster_vectors[c1], cluster_vectors[c2]
        top_vars.append(variables[np.argmax(np.abs(v1 - v2))])
        cluster_vectors[len(profile_scaled) + i] = (v1 + v2) / 2

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 7), facecolor="white")
    ax.set_facecolor("white")

    ddata = dendrogram(
        Z,
        labels                = [f"C{c}" for c in profile.index],
        ax                    = ax,
        color_threshold       = 0,
        above_threshold_color = "steelblue",
        leaf_font_size        = 13,
        leaf_rotation         = 0,
    )

    # Annotate each merge node with the most discriminating variable
    for i, (x_coords, y_coords) in enumerate(zip(ddata["icoord"], ddata["dcoord"])):
        x_mid = (x_coords[1] + x_coords[2]) / 2
        y_mid =  y_coords[1]
        ax.text(
            x_mid, y_mid + 0.01,
            top_vars[i],
            ha         = "center",
            va         = "bottom",
            fontsize   = 8,
            color      = "crimson",
            fontstyle  = "italic",
            rotation   = 30,
            bbox       = dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7),
        )

    ax.set_title(title, fontsize=14, fontweight="bold", pad=15)
    ax.set_ylabel("Ward distance (normalized)", fontsize=11)
    ax.set_xlabel("Cluster", fontsize=11)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")  # ← savefig AVANT show
    plt.show()
    plt.close()
    log.info(f"Saved: {path}")

# ==============================================================================
# FUNCTION 7: UMAP & t-SNE visualizations (2D png + 3D interactive HTML)
# ==============================================================================

def build_cluster_palette(labels):
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    palette   = (sns.color_palette("tab20", len(unique_clusters)) if len(unique_clusters) <= 20
                 else sns.color_palette("hsv", len(unique_clusters)))
    color_map = {c: palette[i] for i, c in enumerate(unique_clusters)}
    color_map[-1] = "lightgrey"
    return color_map


def _to_hex(color_map, c):
    col = color_map[c]
    return "#d3d3d3" if col == "lightgrey" else mcolors.to_hex(col)


def _scatter_clusters(emb, labels, color_map, title, out_dir, filename):
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    fig, ax = plt.subplots(figsize=(9, 7), facecolor="white")
    ax.set_facecolor("white")
    mask_noise = labels == -1
    if mask_noise.any():
        ax.scatter(emb[mask_noise, 0], emb[mask_noise, 1],
                   c="lightgrey", s=8, linewidth=0, label="Outliers (-1)", zorder=1, alpha=0.5)
    for c in unique_clusters:
        mask = labels == c
        ax.scatter(emb[mask, 0], emb[mask, 1], color=color_map[c],
                   s=10, linewidth=0, label=f"C{c}", zorder=2, alpha=0.8)
    ax.set_title(title)
    ax.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left",
              markerscale=2, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=150, facecolor="white")
    plt.close()


def plot_umap_2d(X, labels, title, out_dir, filename, metric="precomputed"):
    os.makedirs(out_dir, exist_ok=True)
    emb       = umap.UMAP(n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
                           min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    color_map = build_cluster_palette(labels)
    _scatter_clusters(emb, labels, color_map, title, out_dir, filename)
    return color_map


def plot_umap_3d_html(X, labels, title, out_dir, filename_html, metric="precomputed", color_map=None):
    os.makedirs(out_dir, exist_ok=True)
    emb = umap.UMAP(n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
                    min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    df_plot = pd.DataFrame({"x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
                             "label": [str(l) for l in labels]})
    df_plot["order"] = df_plot["label"].apply(lambda x: -1 if x == "-1" else int(x))
    df_plot = df_plot.sort_values("order")
    fig = px.scatter_3d(df_plot, x="x", y="y", z="z", color="label",
                        color_discrete_map={str(c): _to_hex(color_map, c) for c in list(unique_clusters) + [-1]},
                        title=title, opacity=0.8,
                        category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]})
    fig.update_traces(marker=dict(size=3))
    fig.update_layout(legend_title_text="Cluster")
    fig.write_html(os.path.join(out_dir, filename_html), include_plotlyjs="cdn")
    log.info(f"Saved: {filename_html}")


def plot_tsne_2d(X, labels, title, out_dir, filename, metric="precomputed", color_map=None):
    os.makedirs(out_dir, exist_ok=True)
    init = umap.UMAP(n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
                     min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    emb  = TSNE(n_components=2, perplexity=VIZ_TSNE_PERP, learning_rate="auto",
                init=init, metric=metric, max_iter=VIZ_TSNE_ITER).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    _scatter_clusters(emb, labels, color_map, title, out_dir, filename)


def plot_tsne_3d_html(X, labels, title, out_dir, filename_html, metric="precomputed", color_map=None):
    os.makedirs(out_dir, exist_ok=True)
    init = umap.UMAP(n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
                     min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    emb  = TSNE(n_components=3, perplexity=VIZ_TSNE_PERP, learning_rate="auto",
                init=init, metric=metric, max_iter=VIZ_TSNE_ITER).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    df_plot = pd.DataFrame({"x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
                             "label": [str(l) for l in labels]})
    fig = px.scatter_3d(df_plot, x="x", y="y", z="z", color="label",
                        color_discrete_map={str(c): _to_hex(color_map, c) for c in list(unique_clusters) + [-1]},
                        title=title, opacity=0.8,
                        category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]})
    fig.update_traces(marker=dict(size=3))
    fig.update_layout(legend_title_text="Cluster")
    fig.write_html(os.path.join(out_dir, filename_html), include_plotlyjs="cdn")
    log.info(f"Saved: {filename_html}")


# ==============================================================================
# FUNCTION 8: Outlier description
# Heatmap + bar chart comparing outliers vs clustered patients
# ==============================================================================

def describe_outliers_internal(df_sub, labels, run_label, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    df_work            = df_sub.copy()
    df_work["cluster"] = labels
    df_noise           = df_work[df_work["cluster"] == -1]
    df_clustered       = df_work[df_work["cluster"] != -1]
    n_noise            = len(df_noise)
    n_total            = len(df_work)
    if n_noise == 0:
        return
    numeric_cols = [c for c in df_sub.select_dtypes(include="number").columns]
    compare = pd.DataFrame({
        "outliers":  df_noise[numeric_cols].mean().round(3),
        "clustered": df_clustered[numeric_cols].mean().round(3),
    })
    compare["diff"] = (compare["outliers"] - compare["clustered"]).round(3)
    compare = compare.sort_values("diff", key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 4), facecolor="white")
    ax.set_facecolor("white")
    sns.heatmap(compare[["outliers", "clustered"]].T, annot=True, fmt=".2f",
                cmap="YlOrRd", ax=ax, annot_kws={"size": 8})
    ax.set_title(f"Outliers vs. clustered - {run_label}")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "outliers_heatmap.png"), dpi=150,
                bbox_inches="tight", facecolor="white")
    plt.close()

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 5), facecolor="white")
    ax.set_facecolor("white")
    colors = ["tab:red" if v > 0 else "tab:blue" for v in compare["diff"]]
    ax.bar(compare.index, compare["diff"], color=colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Difference (outliers - clustered)")
    ax.set_title(f"Outlier vs. clustered differences - {run_label}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "outliers_diff.png"), dpi=150, facecolor="white")
    plt.close()

    compare.to_csv(os.path.join(out_dir, "outliers_compare.csv"))
    log.info(f"[{run_label}] Outlier description done.")


# ==============================================================================
# FUNCTION 9: Full postprocessing
# Calls all visualization functions for a final run
# ==============================================================================

def run_full_postprocessing(df_sub, D, fit_input, labels, clusterer,
                            run_label, out_dir, metric="precomputed"):
    """
    Full visualization suite:
    1. HDBSCAN native trees (condensed, single linkage, MST, persistence)
    2. Cluster profile heatmap + Ward dendrogram
    3. UMAP 2D/3D + t-SNE 2D/3D
    4. Outlier description (heatmap, bar chart, CSV)
    """
    t_start = time.time()
    os.makedirs(out_dir, exist_ok=True)

    plot_hdbscan_trees(clusterer, run_label, os.path.join(out_dir, "hdbscan_trees"))

    profile = plot_cluster_heatmap(df_sub, labels,
                                   title=f"Cluster profiles - {run_label}",
                                   out_dir=out_dir, filename="heatmap.png")
    if len(profile) >= 2:
        plot_cluster_dendrogram(profile, title=f"Cluster dendrogram - {run_label}",
                                out_dir=out_dir, filename="dendrogram.png")

    viz_input  = fit_input if metric == "euclidean" else D
    viz_metric = metric

    color_map = plot_umap_2d(viz_input, labels, title=f"UMAP 2D - {run_label}",
                             out_dir=out_dir, filename="umap2d.png", metric=viz_metric)
    plot_umap_3d_html(viz_input, labels, title=f"UMAP 3D - {run_label}",
                      out_dir=out_dir, filename_html="umap3d.html",
                      metric=viz_metric, color_map=color_map)
    plot_tsne_2d(viz_input, labels, title=f"t-SNE 2D - {run_label}",
                 out_dir=out_dir, filename="tsne2d.png",
                 metric=viz_metric, color_map=color_map)
    plot_tsne_3d_html(viz_input, labels, title=f"t-SNE 3D - {run_label}",
                      out_dir=out_dir, filename_html="tsne3d.html",
                      metric=viz_metric, color_map=color_map)

    describe_outliers_internal(df_sub, labels, run_label,
                               os.path.join(out_dir, "outliers"))

    log.info(f"[{run_label}] Full postprocessing done in {time.time()-t_start:.1f}s")


print("All functions loaded successfully.")


All functions loaded successfully.


---
## CHUNK 3 - STEP 1: Distance matrix + 2D grid sweep

**Goal:** Jointly identify the optimal (mcs, min_samples) pair for each run.

**What happens here:**
- Distance matrix computed once per run (stored for reuse)
- 2D grid sweep over (mcs, ms) - all values as % of N
- 4 heatmaps per run: outlier rate, silhouette, combined score, n_clusters
- Top 10 configurations by lowest outlier rate and combined score

**After running this chunk:**
1. Look at the heatmaps - find low outlier rate + reasonable n_clusters
2. Update FINAL_PARAMS in Chunk 1
3. Run Chunk 4

**Tip:** Try `HDBSCAN_CLUSTER_METHOD = 'leaf'` in Chunk 1 to reduce outliers.


In [39]:
# ==============================================================================
# STEP 1: Load data  + compute distance matrices + 2D grid sweep
# ==============================================================================


#-------------------------------------------
# load data
#-------------------------------------------
df = load_and_preprocess(CSV_PATH)
N  = len(df)
print(f"Dataset loaded: {N:,} patients, {df.shape[1]} variables")

print(f"\nGrid parametrization for N={N:,}:")
for pct in MCS_PCT_VALUES:
    mcs     = int(N * pct)
    ms_vals = sorted(set(MS_FIXED_VALUES + [max(1, int(mcs * p)) for p in MS_PCT_MCS_VALUES]))
    print(f"  mcs={mcs:5d} ({pct*100:.1f}%N) | ms={ms_vals}")


INFO | 
bio_exam_count distribution:
 bio_exam_count     n  pct
              0 23023 40.5
              1 23541 41.5
              2  7936 14.0
              3  2187  3.9
              4    97  0.2
INFO | 
imaging_exam_count distribution:
 imaging_exam_count     n  pct
                  0 27258 48.0
                  1 24924 43.9
                  2  4190  7.4
                  3   375  0.7
                  4    36  0.1
                  5     1  0.0


Dataset loaded: 56,784 patients, 191 variables

Grid parametrization for N=56,784:
  mcs= 1135 (2.0%N) | ms=[1, 5, 10, 11, 15, 56, 113, 227]
  mcs= 1703 (3.0%N) | ms=[1, 5, 10, 15, 17, 85, 170, 340]
  mcs= 2271 (4.0%N) | ms=[1, 5, 10, 15, 22, 113, 227, 454]
  mcs= 2839 (5.0%N) | ms=[1, 5, 10, 15, 28, 141, 283, 567]
  mcs= 3407 (6.0%N) | ms=[1, 5, 10, 15, 34, 170, 340, 681]


In [ ]:

#-------------------------------------------
# compute distance matrices + 2D grid sweep
#-------------------------------------------
# SCENARIO 2 - Hamming + Manhattan, MinMax vs Standard scaler
S2_RUNS = [
    #{
    #     "run_label":    "s2_noweights",
    #     "col_weights":  None,
    #     "weight_quanti": 1.0,
    # },
    {
        "run_label":    "s2_balanced",
        "weight_quanti": 0.5,
        "col_weights": {
            "has_ultrasound":      0.25,
            "has_ct_scan":         0.25,
            "has_xray":            0.25,
            "has_mri":             0.25,
            "has_blood_test":      0.20,
            "has_culture":         0.20,
            "has_lumbar_puncture": 0.20,
            "has_blood_gas":       0.20,
            "had_ekg":             0.20,
            "hospitalization":     0.50,
            "observation_unit":    0.50,
        },
    },
]

stored = {}
for scaler in SCALERS:
    print(f"\n{'='*60}\nSCALER: {scaler}\n{'='*60}")

    for run in S2_RUNS:
        label   = f"{run['run_label']}_{scaler}"
        out_dir = os.path.join(OUTPUT_DIR, scaler, run["run_label"])
        print(f"\n--- {label} ---")

        df_sub, D, fit_input, labels, clusterer = run_hdbscan(
            df = df,
            scenario_name   = "scenario_2",
            run_label       = run["run_label"],
            distance_metric = "precomputed",
            scaler          = scaler,
            col_weights     = run["col_weights"],
            weight_quanti   = run["weight_quanti"],
        )

        stored[label] = {"df_sub": df_sub, "fit_input": D, "metric": "precomputed"}

        run_grid_sweep(fit_input=D, metric="precomputed",
                       run_label=label, out_dir=out_dir, N=N)
#
#
# # SCENARIO 3 - Gower -> UMAP -> grid sweep
# S3_RUNS = [
#     {"run_label": "s3_noweights","gower_weights": None},
#     {"run_label": "s3_catdown", "gower_weights": {
#             **{col: WEIGHT_BIO_VEINOUS      for col in BIO_VEINOUS},
#             **{col: WEIGHT_IMAGING_DETAILED for col in IMAGING_COLS_DETAILED},
#         },
#     },
#
#     {"run_label"    : "s3_allblocks_balanced","gower_weights": {
#             **{col: WEIGHT_BIO_VEINOUS      for col in BIO_VEINOUS},
#             **{col: WEIGHT_IMAGING_DETAILED for col in IMAGING_COLS_DETAILED},
#             **{col: WEIGHT_IMAGING_BOOL     for col in IMAGING_COLS_BOOL},
#             **{col: WEIGHT_BIO_EXAMS        for col in BIO_EXAMS},
#             **{col: WEIGHT_DISPOSITION      for col in DISPOSITION_COLS},
#             **{col: WEIGHT_PROCEDURES       for col in PROCEDURE_COLS},
#         },
#     },
# ]
#
# print(f"\n{'='*60}\nSCENARIO 3 - Gower\n{'='*60}")
#
# stored = {}
# for run in S3_RUNS:
#     label   = run["run_label"]
#     out_dir = os.path.join(OUTPUT_DIR, "gower", label)
#     print(f"\n--- {label} ---")
#
#     df_sub, D, fit_input, labels, clusterer = run_hdbscan(
#         df = df,
#         scenario_name   = "scenario_3",
#         run_label       = label,
#         distance_metric = "gower",
#         gower_weights   = run["gower_weights"],
#     )
#
#     stored[label] = {"df_sub": df_sub, "fit_input": fit_input, "metric": "euclidean"}
#
#     run_grid_sweep(fit_input=fit_input, metric="euclidean",
#                    run_label=label, out_dir=out_dir, N=N)
#
#
#

print("\n" + "="*60)
print("STEP 1 COMPLETE")
print("="*60)
print("-> Review the 4 curves figures per run above")
print("-> Pick (mcs, ms): low outlier rate + stable clusters")
print("-> Update FINAL_PARAMS in Chunk 1")
print("-> Run Chunk 4")


INFO | [s2_balanced] n=56784 | binary=11 | cat=0 | quanti=2



SCALER: minmax

--- s2_balanced_minmax ---


S2 no weight => Dans contexte clinique — profils de patients aux urgences — un ms élevé comme 170 n'est pas forcément un problème si clusters sont bien interprétables. Ce qui compte c'est la cohérence clinique des groupes obtenus, pas la valeur de ms en elle-même.
Je testerais les deux (ms=5 et ms=170) pour mcs=3406 et je comparerais les signatures des clusters — si les 6 clusters de ms=170 sont cliniquement plus cohérents que les 6 de ms=34, c'est lui qu'il faut garder.


Meilleurs resultats avec balanced pour scenario 2!!!!!!!


## S2 Noweights vs S2 Balanced — Justification for Weighted Approach

### Methodological rationale for block-weighting

In `s2_noweights`, all binary variables contribute equally with a weight of 1.0.
However, Scenario 2 contains 11 binary variables distributed across clinically distinct
blocks (4 imaging flags, 4 biology flags, 1 EKG, 2 disposition variables), plus 2
quantitative variables. Without weighting, the biology and imaging blocks collectively
dominate the distance matrix simply by their number of variables — not because they are
clinically more informative. This introduces a structural bias unrelated to clinical
relevance.

`s2_balanced` corrects this by assigning each block an equal total contribution of 1.0,
regardless of the number of variables it contains — the same logic applied by Gower
weighting in Scenario 3. Weights were defined **a priori** based on clinical reasoning,
before any sweep results were observed.

### Quantitative comparison

| Metric | s2_noweights (best) | s2_balanced (best) |
|---|---|---|
| Max silhouette | 0.424 | 0.507 |
| Min outlier rate | 2.0% | 2.3% |
| Max combined score | 0.470 | 0.457 |
| Stability | 1.0 (all) | 1.0 (all) |

The balanced weighting yields a consistently higher silhouette across all configurations
(+0.05 to +0.08 on average), indicating better cluster separation. Stability remains
perfect (1.0) in both runs, confirming that the underlying cluster structure is robust
regardless of weighting. The slight difference in combined scores is explained by the
penalization of higher outlier rates at small mcs values in the balanced run.

### Conclusion

`s2_balanced` is retained as the primary Scenario 2 configuration. The block-weighting
approach is methodologically sounder and produces better-separated clusters, as reflected
by the consistently higher silhouette scores. Importantly, weights were defined a priori
based on clinical reasoning — not optimized to maximize metrics — which protects against
post-hoc bias. This strategy is consistent with the explicit block-weighting applied in
`s3_allblocks_balanced`, where the same principle was used to prevent the venous biology
block (27 variables) from dominating the Gower distance matrix by sheer variable count.
In both scenarios, the goal is the same: ensure that each clinical dimension contributes
equitably to the inter-patient distance, independent of how many variables represent it.

---
## CHUNK 4 - STEP 2: Final run with optimal parameters + full visualizations

**Prerequisites:**
1. FINAL_PARAMS updated in Chunk 1
2. Either stored in memory (from Chunk 3) OR cache file available

**Visualizations per run:**
- HDBSCAN native trees: condensed, single linkage, MST (S3 only), cluster persistence
- Cluster profile heatmap + Ward dendrogram (most discriminating variable per merge)
- UMAP 2D (png) + UMAP 3D (interactive HTML)
- t-SNE 2D (png) + t-SNE 3D (interactive HTML)
- Outlier description: heatmap, bar chart, CSV


## Scenario 2 balanced

In [ ]:
run_label = "s2_balanced"
scaler    = "minmax"

df = load_and_preprocess(CSV_PATH)
N  = len(df)
print(f"Dataset loaded: {N:,} patients, {df.shape[1]} variables")


# -------------------------------------------------------------------------------------------
# UPDATE AFTER CHUNK 3
FINAL_PARAMS = {
    "s2_balanced_minmax"   : (0.06, 0.01),
}
# Conversion % -> valeurs absolues
mcs_pct, ms_pct = FINAL_PARAMS[f"{run_label}_{scaler}"]
mcs = max(10, int(N * mcs_pct))
ms  = max(1,  int(mcs * ms_pct))
print(f"N={N:,} → mcs={mcs:,} | ms={ms}")

# 1. Charger la matrice déjà calculée
npz_path = os.path.join(OUTPUT_DIR, scaler, run_label, f"distance_matrix_{run_label}_latest.npz")
D        = np.load(npz_path)["D"]

# 2. Reconstruire df_sub
cols   = [c for c in SCENARIOS["scenario_2"] if c in df.columns]
df_sub = df[cols].copy().dropna()

# 3. Lancer HDBSCAN
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=mcs, min_samples=ms,
    metric="precomputed", cluster_selection_method=HDBSCAN_CLUSTER_METHOD,
    gen_min_span_tree=True,
).fit(D)

labels          = clusterer.labels_
unique_clusters = np.unique(labels[labels != -1])

# 4. Taille des clusters + outliers
print(f"\nN total    : {len(labels):,}")
print(f"N outliers : {(labels == -1).sum():,} ({100*(labels == -1).mean():.1f}%)")
print(f"N clusters : {len(unique_clusters)}\n")
print("Taille par cluster :")
for c in unique_clusters:
    n = (labels == c).sum()
    print(f"  Cluster {c} : {n:,} ({100*n/len(labels):.1f}%)")

# 5. Cluster le plus proche des outliers
outlier_idx = np.where(labels == -1)[0]
mean_dists  = np.stack([
    D[outlier_idx][:, labels == c].mean(axis=1)
    for c in unique_clusters
], axis=1)
nearest_cluster = unique_clusters[mean_dists.argmin(axis=1)]
df_nearest = pd.DataFrame({
    "nearest_cluster": nearest_cluster,
    "mean_dist":       mean_dists.min(axis=1).round(4)
})
print(f"\nCluster le plus proche des outliers :")
print(df_nearest["nearest_cluster"].value_counts())

# 6. Sauvegarder CSV
out_dir = os.path.join(OUTPUT_DIR, scaler, run_label, f"final_mcs{mcs}_ms{ms}")
os.makedirs(out_dir, exist_ok=True)
df_out            = df.loc[df_sub.index].copy()
df_out["cluster"] = labels
df_out.to_csv(os.path.join(out_dir, f"clustering_{run_label}_{scaler}.csv"), index=True)
print(f"\nCSV sauvegardé : {out_dir}")

# 7. Postprocessing (long)
run_full_postprocessing(
    df_sub=df_sub, D=D, fit_input=D,
    labels=labels, clusterer=clusterer,
    run_label=f"{run_label}_{scaler}",
    out_dir=out_dir,
    metric="precomputed",
)

## MS 170 & 34##

Cluster 0 — Hospitalisés avec bilan complet

hospitalization=1, has_blood_test=0.85, bio_exam_count=1.09, imaging_exam_count=0.82
Patients hospitalisés avec beaucoup d'examens biologiques et d'imagerie

Cluster 1 — Observation + bilan lourd

observation_unit=1, hospitalization=1, has_blood_test=0.95, bio_exam_count=1.54
Patients passés en UHCD avec le bilan le plus complet de tous les clusters

Cluster 2 — Passage simple avec biologie

observation_unit=0, hospitalization=0, has_blood_test=0.90, bio_exam_count=1.14
Patients rentrés à domicile avec bilan biologique mais peu d'imagerie

Cluster 3 — Passage minimal

Tout à 0 ou quasi-0
Patients sans examen, probablement retour à domicile rapide

Cluster 4 — Imagerie échographique isolée

has_ultrasound=1.00, imaging_exam_count=1.02, tout le reste à 0
Patients venus uniquement pour une échographie, sans bilan biologique



"Le scénario 3 a été évalué mais présente une stabilité des clusters nettement inférieure (≈0.20 vs 1.0 pour S2), indiquant que les structures identifiées dépendent fortement des hyperparamètres et ne sont pas reproductibles. S2 offre par ailleurs une meilleure traçabilité métier des distances utilisées. Le scénario 2 est donc retenu pour l'analyse des profils."

In [ ]:
run_label = "s2_balanced"
scaler = "minmax"

df = load_and_preprocess(CSV_PATH)
N = len(df)
print(f"Dataset loaded: {N:,} patients, {df.shape[1]} variables")

# -------------------------------------------------------------------------------------------
# UPDATE AFTER CHUNK 3
FINAL_PARAMS = {
    "s2_balanced_minmax": (0.04, 0.00666),
}
# Conversion % -> valeurs absolues
mcs_pct, ms_pct = FINAL_PARAMS[f"{run_label}_{scaler}"]
mcs = max(10, int(N * mcs_pct))
ms = max(1, int(mcs * ms_pct))
print(f"N={N:,} → mcs={mcs:,} | ms={ms}")

# 1. Charger la matrice déjà calculée
npz_path = os.path.join(OUTPUT_DIR, scaler, run_label, f"distance_matrix_{run_label}_latest.npz")
D = np.load(npz_path)["D"]

# 2. Reconstruire df_sub
cols = [c for c in SCENARIOS["scenario_2"] if c in df.columns]
df_sub = df[cols].copy().dropna()

# 3. Lancer HDBSCAN
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=mcs, min_samples=ms,
    metric="precomputed", cluster_selection_method=HDBSCAN_CLUSTER_METHOD,
    gen_min_span_tree=True,
).fit(D)

labels = clusterer.labels_
unique_clusters = np.unique(labels[labels != -1])

# 4. Taille des clusters + outliers
print(f"\nN total    : {len(labels):,}")
print(f"N outliers : {(labels == -1).sum():,} ({100 * (labels == -1).mean():.1f}%)")
print(f"N clusters : {len(unique_clusters)}\n")
print("Taille par cluster :")
for c in unique_clusters:
    n = (labels == c).sum()
    print(f"  Cluster {c} : {n:,} ({100 * n / len(labels):.1f}%)")

# 5. Cluster le plus proche des outliers
outlier_idx = np.where(labels == -1)[0]
mean_dists = np.stack([
    D[outlier_idx][:, labels == c].mean(axis=1)
    for c in unique_clusters
], axis=1)
nearest_cluster = unique_clusters[mean_dists.argmin(axis=1)]
df_nearest = pd.DataFrame({
    "nearest_cluster": nearest_cluster,
    "mean_dist": mean_dists.min(axis=1).round(4)
})
print(f"\nCluster le plus proche des outliers :")
print(df_nearest["nearest_cluster"].value_counts())

# 6. Sauvegarder CSV
out_dir = os.path.join(OUTPUT_DIR, scaler, run_label, f"final_mcs{mcs}_ms{ms}")
os.makedirs(out_dir, exist_ok=True)
df_out = df.loc[df_sub.index].copy()
df_out["cluster"] = labels
df_out.to_csv(os.path.join(out_dir, f"clustering_{run_label}_{scaler}.csv"), index=True)
print(f"\nCSV sauvegardé : {out_dir}")

# 7. Postprocessing (long)
run_full_postprocessing(
    df_sub=df_sub, D=D, fit_input=D,
    labels=labels, clusterer=clusterer,
    run_label=f"{run_label}_{scaler}",
    out_dir=out_dir,
    metric="precomputed",
)